#Упражнения

## Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

тк ```map``` генератор, получаем весь результат через ```list```.

Он вернет список пар/кортежей и при попытке это все закинуть в список получится вложенность списка в список, поэтому передаем все элементы результирующего списка в новый с ```extend```
        ```mapped.extend(list(MAP(k, v)))```

```shaffle``` при нахождении ключа, которого еще нет в словаре, cоздает пустой список для него, в который и закинет все значения с таким ключом

Итог: все числа рассортированы по своим "биркам"

после формирования словаря ключ-список значений, мы переходим к ```reduce```.

```for k, vs in groups.items():``` - к - ключ, vs - список значений

```final.extend(list(REDUCE(k, vs)))``` -  ```reduce``` возвращает список кортежей пар ключ - нужное значение (например max) из переданного списка


In [ ]:

def MapReduce(RECORDREADER, MAP, REDUCE):
    # 1. Читаем данные
    data = list(RECORDREADER())

    # 2. MAP
    mapped = []
    for k, v in data:
        mapped.extend(list(MAP(k, v)))

    # 3. Shuffle
    groups = {}
    for k, v in mapped:
        if k not in groups: groups[k] = []
        groups[k].append(v)


    # 4. REDUCE
    final = []
    for k, vs in groups.items():
        final.extend(list(REDUCE(k, vs)))
    return final

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
def RECORDREADER():
  # в нашем заднии можно обойтись return, но если 9348098409093 чисел, то лучше выдавать партиями
    yield (None, [1, 42, 17, 99, 23])

def MAP(_, numbers):
    for n in numbers:
      #тут могло бы быть условие для отбора чисел
      #и только на них мы бы приклеили бирку
        yield ("max", n)

def REDUCE(key, values):
    yield (key, max(values))


In [ ]:
output = MapReduce(RECORDREADER, MAP, REDUCE)
list(output)

[('max', 99)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def RECORDREADER():
    yield (None, [10, 20, 30, 40, 50])

def MAP(_, numbers):
    for n in numbers:
        yield ("avg_calculation", n)

def REDUCE(key, values):
    res = sum(values) / len(values)
    yield (key, res)



In [ ]:
print(MapReduce(RECORDREADER, MAP, REDUCE))

[('avg_calculation', 30.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def SHUFFLE(mapped_items):
    # 1. Сортируем список по ключам
    sorted_items = sorted(mapped_items)

    if not sorted_items:
        return


    current_key = sorted_items[0][0] #нулевой кортеж нулевое значение наш первый ключ
    current_values = []


    for k, v in sorted_items:
        if k == current_key:
            current_values.append(v) #ключ совпал - кидаем в список
        else:
            # Если ключ сменился, отдаем накопленную группу
            yield (current_key, current_values)


            current_key = k
            current_values = [v]


    yield (current_key, current_values)

In [ ]:

test_data = [
    ('яблоко', 1),
    ('банан', 10),
    ('яблоко', 2),
    ('груша', 5),
    ('банан', 20)
]


result = list(SHUFFLE(test_data))

for key, values in result:
    print(f"Ключ: {key}, Группа: {values}")


Ключ: банан, Группа: [10, 20]
Ключ: груша, Группа: [5]
Ключ: яблоко, Группа: [1, 2]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
def RECORDREADER():
    yield (None, ["apple", "banana", "apple", "orange", "banana", "banana"])

def MAP(_, items):
    for item in items:
        # Сам элемент становится ключом.
        yield (item, None)

def REDUCE(key, values):
    yield key

In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

['apple', 'banana', 'orange']

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



создаем пару (t,t), тк MApReduce работает с парами ключ-значение. Поэтому сам кортеж становится и ключом и значением.

Следовательно если буду дубликаты кортежей, то shuffle их закинет в одну группу по одиинаковому ключу, а reduce вернет только сам ключ и не будет никаких дубликатов

In [ ]:
def RECORDREADER():
    users = [
        (1, "Ivan", 20),
        (2, "Oleg", 15),
        (3, "Dmitry", 30),
        (4, "Anna", 12)
    ]
    yield (None, users)

def MAP(_, users_list):
    for t in users_list:  #t — это весь кортеж (1, "Ivan", 20)
        if t[2] >= 18:
            # Если условие истинно, создаем пару (t, t)
            yield (t, t)

def REDUCE(key, values):
    # Функция идентичности. На вход пришла пара ключ (кортеж) - значение (тот же кортеж).
    # Мы просто возвращаем ключ.
    yield key

In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

[(1, 'Ivan', 20), (3, 'Dmitry', 30)]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
def RECORDREADER():
    #ID, Имя, Направление
    data = [
        (101, "Иванов", "ФИИТ"),
        (102, "Петров", "ИВТ"),
        (103, "Сидоров", "Экономика"),
        (104, "Кузнецов", "Реклама")
    ]
    yield (None, data)

def MAP(_, table_rows):#table_rows - весь список кортежей
    # S = {Направление}
    for t in table_rows:
        # 1. Создаем t' — кортеж только с нужными атрибутами
        t_prime = (t[2],)
        yield (t_prime, t_prime)

def REDUCE(t_prime, values):
    yield t_prime

In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

[('ФИИТ',), ('ИВТ',), ('Экономика',), ('Реклама',)]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
def RECORDREADER():
    R = [("Иван", 20), ("Олег", 22)]
    S = [("Иван", 20), ("Алла", 25)]
    yield (None, R + S)

def MAP(_, combined_rows):
    for t in combined_rows:
        yield (t, t)

def REDUCE(t, values):
    yield t

In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

[('Иван', 20), ('Олег', 22), ('Алла', 25)]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

После shaffle в reduce попадает ключ-список значений, поэтому мы проверяем два ли значения в списке (значения пришли из обеих таблиц), но только при условии, что таблицы это именно множества без дубликатов, иначе надо будет возвращать в паре с именем таблицы...

In [ ]:
def RECORDREADER():
    R = [("Иван", 20), ("Олег", 22)]
    S = [("Иван", 20), ("Алла", 25)]
    yield (None, R + S)

def MAP(_, combined_rows):
    for t in combined_rows:
        yield (t, t)

def REDUCE(t, values):
    if len(values) == 2:
        yield (t, t)


In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

[(('Иван', 20), ('Иван', 20))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
def RECORDREADER():
    R = [("Иван", 20), ("Олег", 22), ("Дмитрий", 30)]
    S = [("Иван", 20), ("Анна", 25)]

    yield ("R", R)
    yield ("S", S)

def MAP(table_name, rows):
    for t in rows:
        yield (t, table_name)

def REDUCE(t, values):
    if values == ["R"]:
        yield (t, t)


In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

[(('Олег', 22), ('Олег', 22)), (('Дмитрий', 30), ('Дмитрий', 30))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
def RECORDREADER():
    # R (a, b)
    R = [("Иванов", 10), ("Петров", 10), ("Сидоров", 20)]
    # S (b, c)
    S = [(10, "Москва"), (20, "Питер")]
    yield ("R", R)
    yield ("S", S)

def MAP(name, rows):
    for t in rows:
        if name == "R":
            a, b = t
            yield (b, ("R", a))
        elif name == "S":
            b, c = t
            yield (b, ("S", c))

def REDUCE(b, values):
    # values — это список типа [('R', 'Иванов'), ('R', 'Петров'), ('S', 'Москва')]


    list_R = [v[1] for v in values if v[0] == "R"] # Тут будут все 'a'
    list_S = [v[1] for v in values if v[0] == "S"] # Тут будут все 'c'


    for a in list_R:
        for c in list_S:

            yield (None, (a, b, c))

In [ ]:
MapReduce(RECORDREADER, MAP, REDUCE)

[(None, ('Иванов', 10, 'Москва')),
 (None, ('Петров', 10, 'Москва')),
 (None, ('Сидоров', 20, 'Питер'))]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
def RECORDREADER():
    # Менеджер, Сумма, Дата
    data = [
        ("Иванов", 100, "01.03"),
        ("Петров", 200, "01.03"),
        ("Иванов", 150, "02.03"),
        ("Петров", 50,  "02.03"),
        ("Сидоров", 300, "02.03")
    ]
    yield (None, data)

def MAP(_, rows):
    for a, b, c in rows:
        # По заданию: для кортежа (a,b,c) создайте пару (a,b)
        yield (a, b)

def REDUCE(a, values,theta):
    """
    a: ключ (группа)
    values: список [b1, b2, ..., bn]
    theta: агрегирующая операция (функция)
    """
    # Применяем агрегирующую операцию theta к списку значений
    x = theta(values)
    yield (a, x)

In [ ]:
# Добавляем theta в аргументы движка
def MapReduce(RECORDREADER, MAP, REDUCE, theta):
    # 1. Читаем данные
    data = list(RECORDREADER())

    # 2. MAP
    mapped = []
    for k, v in data:
        mapped.extend(list(MAP(k, v)))

    # 3. Shuffle
    groups = {}
    for k, v in mapped:
        if k not in groups:
            groups[k] = []
        groups[k].append(v)

    # 4. REDUCE
    final = []
    for k, vs in groups.items():
        final.extend(list(REDUCE(k, vs, theta)))

    return final


In [ ]:

result = MapReduce(RECORDREADER, MAP, REDUCE, sum)
print(result)

result = MapReduce(RECORDREADER, MAP, REDUCE, max)
print(result)

[('Иванов', 250), ('Петров', 250), ('Сидоров', 300)]
[('Иванов', 150), ('Петров', 200), ('Сидоров', 300)]


#

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
def RECORDREADER():
    yield ("M", [(0, 0, 1), (0, 1, 2), (1, 0, 3), (1, 1, 4)]) # (i, j, v)
    yield ("N", [(0, 0, 5), (0, 1, 6), (1, 0, 7), (1, 1, 8)]) # (j, k, w)

def MAP(rel, rows):
    for row in rows:
        if rel == "M":
          # Матрица M: (i, j, v)
            i, j, v = row
            yield (j, ("M", i, v))
        else:
          # Матрица N: (j, k, w)
            j, k, w = row
            yield (j, ("N", k, w))

def REDUCE(j, values):#из-за кучи в шаффле, сортируем на данные из М и из Н в пределах одного ключа
    ms = [val for val in values if val[0] == "M"]
    ns = [val for val in values if val[0] == "N"]
    for _, i, v in ms:
        for _, k, w in ns:
            yield ((i, k), v * w)

def MapReduce(RECORDREADER, MAP, REDUCE):
    data = list(RECORDREADER())
    # MAP
    mapped = []
    for k, v in data:
        mapped.extend(list(MAP(k, v)))


    # SHUFFLE (сваливаем все данные из М и Н с одинаковым j в одну кучу)
    groups = {}
    for k, v in mapped:
        if k not in groups:
            groups[k] = []
        groups[k].append(v)

    # Join и перемножение
    intermediate = []
    for k, vs in groups.items():
        intermediate.extend(list(REDUCE(k, vs)))

    # Финальная агрегация по (i, k) — сумма v*w
    res = {}
    for (ik, prod) in intermediate:
        res[ik] = res.get(ik, 0) + prod
    return list(res.items())

print(MapReduce(RECORDREADER, MAP, REDUCE))

[((0, 0), 19), ((0, 1), 22), ((1, 0), 43), ((1, 1), 50)]


Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1


def REDUCE(key, values):
  (i, k) = key


In [ ]:
import numpy as np

# Инициализация размерностей и тестовых матриц
I, J, K = 2, 3, 4
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def RECORDREADER():
    """
    Построчное чтение элементов матрицы big_mat.
    Возвращает координаты (j, k) и значение элемента.
    """
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    """
    Принимает индекс и значение из big_mat.
    Умножает значение на соответствующий столбец small_mat.
    Генерирует промежуточные пары ((i, k), частичное_произведение).
    """
    (j, k) = k1
    w = v1

    for i in range(small_mat.shape[0]):
        m_val = small_mat[i, j]
        yield ((i, k), m_val * w)

def REDUCE(key, values):
    """
    Суммирует все частичные произведения для каждой целевой ячейки (i, k).
    """
    yield (key, sum(values))

def MapReduce(RR, MAP, REDUCE):
    # Стадия MAP
    mapped = []
    for k1, v1 in RR():
        mapped.extend(list(MAP(k1, v1)))

    # Стадия SHUFFLE (группировка по ключу)
    groups = {}
    for k2, v2 in mapped:
        if k2 not in groups:
            groups[k2] = []
        groups[k2].append(v2)

    # Стадия REDUCE
    final = []
    for k3, vs in groups.items():
        final.extend(list(REDUCE(k3, vs)))

    return dict(final)

# Проверка корректности вычислений
result = MapReduce(RECORDREADER, MAP, REDUCE)
expected = np.dot(small_mat, big_mat)

print(f"Результат MapReduce для (0,0): {result[(0, 0)]}")
print(f"Результат NumPy для (0,0):     {expected[0, 0]}")

Результат MapReduce для (0,0): 0.4569307672577285
Результат NumPy для (0,0):     0.4569307672577285


Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution.items()))

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE).items())
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
import numpy as np


def RECORDREADER():
    M = [(0, 0, 1), (0, 1, 2), (1, 0, 3), (1, 1, 4)]
    N = [(0, 0, 5), (0, 1, 6), (1, 0, 7), (1, 1, 8)]
    yield ("M", M)
    yield ("N", N)

def MAP(rel_name, rows):
    for t in rows:
        if rel_name == "M":
            i, j, v = t
            yield (j, ("M", i, v))
        elif rel_name == "N":
            j, k, w = t
            yield (j, ("N", k, w))

def REDUCE(j, values):
    list_M = [v for v in values if v[0] == "M"]
    list_N = [v for v in values if v[0] == "N"]
    for _, i, v in list_M:
        for _, k, w in list_N:
            yield ((i, k), v * v)


def REDUCE(j, values):
    list_M = [v for v in values if v[0] == "M"]
    list_N = [v for v in values if v[0] == "N"]
    for _, i, v in list_M:
        for _, k, w in list_N:
            yield ((i, k), v * w)

# 2. ДВИЖОК
def MapReduce(RR, MAP, REDUCE):
    mapped = []
    for k1, v1 in RR():
        mapped.extend(list(MAP(k1, v1)))
    groups = {}
    for k2, v2 in mapped:
        if k2 not in groups: groups[k2] = []
        groups[k2].append(v2)
    final = []
    for k3, vs in groups.items():
        final.extend(list(REDUCE(k3, vs)))
    return final # Тут возвращаем список кортежей

# 3. ЗАПУСК И ПРОВЕРКА
# Шаг A: Получаем список произведений
intermediate_results = MapReduce(RECORDREADER, MAP, REDUCE)

# Шаг B: Финальная агрегация (сумма)
def FINAL_AGGREGATION(pairs):
    results = {}
    for (key, prod) in pairs:
        results[key] = results.get(key, 0) + prod
    return results

solution_dict = FINAL_AGGREGATION(intermediate_results)

# Сравнение с NumPy
M_mat = np.array([[1, 2], [3, 4]])
N_mat = np.array([[5, 6], [7, 8]])
reference = np.dot(M_mat, N_mat)

print("MapReduce (0,0):", solution_dict[(0, 0)])
print("NumPy (0,0):    ", reference[0, 0])

MapReduce (0,0): 19
NumPy (0,0):     19


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
# Читаем матрицу M
def RECORDREADER_M():
    # Формат: (строка i, столбец j, значение v)
    M = [(0, 0, 1), (0, 1, 2), (1, 0, 3), (1, 1, 4)]
    for item in M:
        yield ("M", item)

# Читаем матрицу N
def RECORDREADER_N():
    # Формат: (строка j, столбец k, значение w)
    N = [(0, 0, 5), (0, 1, 6), (0, 2, 0), (1, 0, 7), (1, 1, 8), (1, 2, 0)]
    for item in N:
        yield ("N", item)

# Объединяющий ридер, который по очереди опрашивает источники
def DISTRIBUTED_RECORDREADER():
    yield from RECORDREADER_M()
    yield from RECORDREADER_N()


def MAP(rel_name, element):
    """
    rel_name: 'M' или 'N'
    element: кортеж (row, col, value)
    """
    i_or_j, j_or_k, val = element

    if rel_name == "M":
        # Ключ - столбец j, значение - метка и индекс строки i
        yield (j_or_k, ("M", i_or_j, val))
    else:
        # Ключ - строка j, значение - метка и индекс столбца k
        yield (i_or_j, ("N", j_or_k, val))

def REDUCE(j, values):
    """
    j: общий индекс
    values: список элементов из обеих матриц, имеющих этот j
    """
    list_M = [v for v in values if v[0] == "M"]
    list_N = [v for v in values if v[0] == "N"]

    # Перемножаем все комбинации элементов, встретившихся по индексу j
    for _, i, v_m in list_M:
        for _, k, v_n in list_N:
            # Результат: координаты будущей матрицы и произведение
            yield ((i, k), v_m * v_n)


# 1. Запускаем основной цикл MapReduce
# Он выдаст список всех произведений v*w для каждой ячейки (i, k)
intermediate_pairs = MapReduce(DISTRIBUTED_RECORDREADER, MAP, REDUCE)

# 2. Суммируем частичные произведения (Stage 2)
def FINAL_AGGREGATION(pairs):
    results = {}
    for (key, prod) in pairs:
        results[key] = results.get(key, 0) + prod
    return results

# Итоговый результат
solution = FINAL_AGGREGATION(intermediate_pairs)

# Проверка (вывод одной ячейки)
print(f"Элемент (0,0): {solution.get((0,0))}")


Элемент (0,0): 19


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

Да, решение будет работать корректно.

Shuffle собирает все элементы с одинаковым ключом $j$ в один REDUCE, независимо от того, из какого ридера и в каком порядке они поступили.

Каждое частичное произведение $M_{ij} \times N_{jk}$ вычисляется автономно.

Коммутативность: Итоговая агрегация — это сумма, а от перемены мест слагаемых сумма не меняется.Главное условие — чтобы через все ридеры в совокупности прошли все ненулевые элементы матриц».

In [ ]:
import numpy as np

# Имитируем несколько источников для матрицы M
def RR_M_part1():
    yield ("M", [(0, 0, 1), (0, 1, 2)]) # Первая строка

def RR_M_part2():
    yield ("M", [(1, 0, 3), (1, 1, 4)]) # Вторая строка

# Имитируем источники для матрицы N
def RR_N_part1():
    yield ("N", [(0, 0, 5), (0, 1, 6)])
def RR_N_part2():
    yield ("N", [(1, 0, 7), (1, 1, 8)])

# Общий распределенный ридер, который объединяет все потоки данных
def MULTI_SOURCE_RECORDREADER():
    sources = [RR_M_part1, RR_M_part2, RR_N_part1, RR_N_part2]
    for source in sources:
        yield from source()

# MAP остается таким же: ключ - общий индекс J
def MAP(rel_name, rows):
    for (r, c, v) in rows:
        if rel_name == "M":
            yield (c, ("M", r, v)) # Ключ - столбец
        else:
            yield (r, ("N", c, v)) # Ключ - строка

# REDUCE делает локальное перемножение пар
def REDUCE(j, values):
    list_M = [v for v in values if v[0] == "M"]
    list_N = [v for v in values if v[0] == "N"]
    for (_, i, v) in list_M:
        for (_, k, w) in list_N:
            yield ((i, k), v * w)

# Финальная сумма
def FINAL_AGGREGATION(pairs):
    results = {}
    for (key, prod) in pairs:
        results[key] = results.get(key, 0) + prod
    return results

# Запуск
intermediate = MapReduce(MULTI_SOURCE_RECORDREADER, MAP, REDUCE)
solution = FINAL_AGGREGATION(intermediate)

print("Элемент (0,0):", solution.get((0,0))) # Должно быть 19 (1*5 + 2*7)

Элемент (0,0): 19
